In [1]:

import random

# Step 1: Define the problem

# Electricity price for each time slot (Rs per kWh)
prices = [8, 6, 4, 3, 5, 9]

# Energy required by each electric vehicle (kWh)
energy = [10, 15, 8, 12, 6]

num_vehicles = len(energy)
num_slots = len(prices)

# Maximum number of vehicles charging in one slot
max_capacity = 2

# Step 2: Initialize parameters
population_size = 10
mutation_rate = 0.1
crossover_rate = 0.8
generations = 100

# Step 3: Create initial population
# Each gene represents the charging time slot of one vehicle

def create_individual():
    individual = []
    for i in range(num_vehicles):
        slot = random.randint(0, num_slots - 1)
        individual.append(slot)
    return individual


def create_population():
    population = []
    for i in range(population_size):
        population.append(create_individual())
    return population


# Step 4: Evaluate fitness
def calculate_cost(individual):
    total_cost = 0
    slot_count = [0] * num_slots

    for i in range(num_vehicles):
        slot = individual[i]
        total_cost += energy[i] * prices[slot]
        slot_count[slot] += 1

    # Peslotnalty if too many vehicles charge in one
    penalty = 0
    for count in slot_count:
        if count > max_capacity:
            penalty += (count - max_capacity) * 100

    return total_cost + penalty


def fitness(individual):
    return 1 / (1 + calculate_cost(individual))



def selection(population):

    candidates = random.sample(population, 3)
    best = candidates[0]

    for candidate in candidates:
        if fitness(candidate) > fitness(best):
            best = candidate

    return best[:]



def crossover(parent1, parent2):
    if random.random() < crossover_rate:
        point = random.randint(1, num_vehicles - 1)

        child1 = parent1[:point] + parent2[point:]
        child2 = parent2[:point] + parent1[point:]

        return child1, child2

    return parent1[:], parent2[:]



def mutation(individual):
    for i in range(num_vehicles):
        if random.random() < mutation_rate:
            individual[i] = random.randint(0, num_slots - 1)

    return individual



random.seed(42)

population = create_population()

best_solution = None
best_cost = float("inf")

for generation in range(generations):


    for individual in population:
        cost = calculate_cost(individual)

        if cost < best_cost:
            best_cost = cost
            best_solution = individual[:]

    new_population = []


    new_population.append(best_solution[:])

    while len(new_population) < population_size:
        parent1 = selection(population)
        parent2 = selection(population)

        child1, child2 = crossover(parent1, parent2)

        child1 = mutation(child1)
        child2 = mutation(child2)

        new_population.append(child1)

        if len(new_population) < population_size:
            new_population.append(child2)

    population = new_population



print("BEST EV CHARGING SCHEDULE")
print("--------------------------------")

total_energy = 0
actual_cost = 0
slot_count = [0] * num_slots

for i in range(num_vehicles):
    slot = best_solution[i]

    print("EV", i + 1,
          "-> Time Slot", slot + 1,
          "| Energy:", energy[i], "kWh",
          "| Price: Rs.", prices[slot], "/kWh")

    total_energy += energy[i]
    actual_cost += energy[i] * prices[slot]
    slot_count[slot] += 1

print("--------------------------------")
print("Total Energy Required:", total_energy, "kWh")
print("Actual Electricity Cost: Rs.", actual_cost)
print("Charging Capacity Penalty: Rs.", best_cost - actual_cost)
print("Best Fitness:", fitness(best_solution))


BEST EV CHARGING SCHEDULE
--------------------------------
EV 1 -> Time Slot 3 | Energy: 10 kWh | Price: Rs. 4 /kWh
EV 2 -> Time Slot 4 | Energy: 15 kWh | Price: Rs. 3 /kWh
EV 3 -> Time Slot 5 | Energy: 8 kWh | Price: Rs. 5 /kWh
EV 4 -> Time Slot 4 | Energy: 12 kWh | Price: Rs. 3 /kWh
EV 5 -> Time Slot 3 | Energy: 6 kWh | Price: Rs. 4 /kWh
--------------------------------
Total Energy Required: 51 kWh
Actual Electricity Cost: Rs. 185
Charging Capacity Penalty: Rs. 0
Best Fitness: 0.005376344086021506
